[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nube1/h-potential-companions/blob/main/methods-series/notebooks/99_reproduce_all.ipynb)

# Reproduce every published number

One table. For each published value: what the papers say, what this package computes now,
the absolute difference, the tolerance, and PASS or FAIL. Closed-form results are held to
the digit. Simulation results are held to an interval wide enough for Monte Carlo error at
the published replicate counts and narrow enough to fail if a method changes.

## Status, before anything else

- **Every datum here is synthetic.** Nothing in this notebook was measured on a person.
- **Papers 1 to 3 are protocols.** They say what will be done, not what was found. No
  participant has been recruited and no Delphi panel has met.
- **Papers 4 to 6 are method illustrations.** Their inputs show a workflow. They are not
  estimates from any health system and not claims about any product.
- **No figure below is evidence about patients.** Each number is a property of a stated
  model or a stated decision rule, and nothing more.

## Setup

The cell below installs the package straight from GitHub when this runs in Colab, and installs the local checkout when it does not. It prints the resolved version so you know what you are running.

**Runtime.** This notebook always runs at full size. Expect about five minutes here and up to ten on a free Colab CPU. Papers 1 and 2 are almost all of it.

In [ ]:
# ---------------------------------------------------------------- setup
# Repository holding this package. The repository is private: in Colab, sign
# in to GitHub or clone it yourself before running.
GITHUB_OWNER = "Nube1"
REPO = "h-potential-companions"
SUBDIR = "methods-series"

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q "hpotential[full] @ git+https://github.com/{GITHUB_OWNER}/{REPO}.git#subdirectory={SUBDIR}"
else:
    # Running from a clone: install the checkout this notebook sits in.
    import importlib.util
    import pathlib
    import subprocess

    if importlib.util.find_spec("hpotential") is None:
        for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (parent / "src" / "hpotential").exists():
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "-e", str(parent)], check=True
                )
                break
        else:
            raise RuntimeError(
                "hpotential is not importable and no local checkout was found. "
                "Run this in Colab with access to the repository, or clone it."
            )

import hpotential

print(f"hpotential {hpotential.__version__}")
print(f"running in Colab: {IN_COLAB}")

In [ ]:
# Importing hpotential selects the Agg backend, because the library writes figures to
# files. Put the plots back into the page, then borrow the series figure style.
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpotential.utils.plotting import GRID, INK, apply_style, series_color

apply_style()
plt.rcParams["figure.dpi"] = 110  # 200 is for print; this is for a screen

try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except Exception:  # Colab sometimes ships without a working widget stack
    HAVE_WIDGETS = False


def explore(fn, controls, defaults):
    """Draw the static case first, then attach sliders if ipywidgets works.

    The static call means the notebook still teaches something when it is read on
    GitHub, where no widget ever runs.
    """
    fn(**defaults)
    if HAVE_WIDGETS:
        display(widgets.interactive(fn, **controls))
    else:
        print("ipywidgets is unavailable here, so the static case above is all there is.")


print("plots inline, style applied, widgets:", HAVE_WIDGETS)

In [ ]:
# ---------------------------------------------------------------- run size
FULL_RUN = True   # True reproduces the published values. This notebook has no reduced mode.


def banner():
    """Print at the top of every results cell. It has to be impossible to miss."""
    if FULL_RUN:
        print("=" * 74)
        print("FULL RUN: published replicate counts. These are the published values.")
        print("=" * 74)
    else:
        print("=" * 74)
        print("REDUCED RUN: cut replicate counts for speed.")
        print("The numbers below are NOT the published values. Set FULL_RUN = True.")
        print("=" * 74)


banner()

## Where the expected values come from

The expected values live in `tests/fixtures/*.json`, the files the published analysis
scripts wrote. Those files ship with the repository but not inside the installed package,
so this notebook looks for them in three places, in order:

1. a local checkout, when the notebook is run from a clone;
2. a shallow clone of the repository, which is what happens in Colab;
3. a small set of values embedded in this notebook, as a last resort.

The third path covers the case where the repository is private or the org name has not been
filled in yet. It is clearly flagged when it fires, because a fixture you carry with you is
weaker evidence than a fixture you fetched.

In [ ]:
import json
import pathlib
import subprocess

FIXTURE_FALLBACK = {
    "p1_determinacy": {"ceiling_rho_D": 0.8778, "c_gamma_Psi": 3.3586,
                       "c_required_for_target": 4.2632, "target": 0.9},
    "p2_pipeline": {"rho_D_population": 0.8361, "beta_true": -0.12,
                    "predicted_attenuated_beta": -0.1003},
    "p3_delphi_oc": {"operating_characteristics": {
        "12": {"false_drop_at_pi_0_90": 0.38075}, "20": {"false_drop_at_pi_0_90": 0.48805},
        "30": {"false_drop_at_pi_0_90": 0.6921}, "45": {"false_drop_at_pi_0_90": 0.76185},
        "60": {"false_drop_at_pi_0_90": 0.80965}}},
    "p4_numerics": {"rk4": {"observed_order": 4.052},
                    "euler_maruyama": {"additive_noise": {"strong_order": 1.03},
                                       "multiplicative_noise": {"strong_order": 0.53}},
                    "fokker_planck": {"grid_convergence_order_L1": 0.9668}},
    "p5_costed": {"psa": {"high-resource": {"P_disagreement": 0.486275}}},
    "p6_voi": {"scenarios": {"central": {"P_cost_effective_QALY": 0.40967,
                                         "per_person_EVPI": 34873.64}}},
}


def find_fixtures():
    """Return (directory, how it was found). Directory is None when embedded values are used."""
    for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        candidate = parent / "tests" / "fixtures"
        if (candidate / "p1_determinacy.json").exists():
            return candidate, "local checkout"
    target = pathlib.Path("/content") / REPO if IN_COLAB else pathlib.Path.cwd() / f"_{REPO}"
    package = target / SUBDIR
    if not (package / "tests" / "fixtures" / "p1_determinacy.json").exists():
        url = f"https://github.com/{GITHUB_OWNER}/{REPO}.git"
        try:
            subprocess.run(["git", "clone", "--depth", "1", url, str(target)],
                           check=True, capture_output=True, timeout=180)
        except Exception as exc:
            print(f"could not clone {url}: {type(exc).__name__}")
    if (package / "tests" / "fixtures" / "p1_determinacy.json").exists():
        return package / "tests" / "fixtures", "shallow clone"
    return None, "embedded fallback"


FIXTURE_DIR, FIXTURE_SOURCE = find_fixtures()


def published(stem):
    if FIXTURE_DIR is None:
        return FIXTURE_FALLBACK[stem]
    return json.loads((FIXTURE_DIR / f"{stem}.json").read_text())


print(f"fixtures from: {FIXTURE_SOURCE}")
if FIXTURE_DIR is None:
    print("!" * 74)
    print("Using values embedded in this notebook, not the published JSON files.")
    print("Clone the repository, or run from a clone, for the real thing.")
    print("!" * 74)
else:
    print(f"directory:     {FIXTURE_DIR}")

## The checking harness

One row per value. `tol` is absolute. Closed-form rows use 1e-6 or tighter; simulation rows
carry a tolerance chosen for the published replicate count and are marked so you can tell
them apart.

In [ ]:
CHECKS = []


def check(paper, what, expected, actual, tol, kind="closed form"):
    diff = abs(float(expected) - float(actual))
    CHECKS.append({"paper": paper, "quantity": what, "kind": kind,
                   "expected": float(expected), "actual": float(actual),
                   "abs diff": diff, "tol": float(tol),
                   "result": "PASS" if diff <= tol else "FAIL"})


def report():
    frame = pd.DataFrame(CHECKS)
    failures = int((frame["result"] == "FAIL").sum())
    print("=" * 96)
    print(f"{len(frame)} checks, {len(frame) - failures} PASS, {failures} FAIL")
    print("=" * 96)
    with pd.option_context("display.width", 200, "display.max_colwidth", 46,
                           "display.float_format", lambda v: f"{v:,.6g}"):
        print(frame.to_string(index=False))
    return frame


print("harness ready")

## Paper 1: determinacy, identification, sample size

In [ ]:
import time

from hpotential.analyses import paper1_validation
from hpotential.core.measurement import (
    determinacy_ceiling, identification_report, required_second_order_signal,
    second_order_determinacy,
)

p1 = published("p1_determinacy")
rho_inf, c = determinacy_ceiling()
check(1, "ceiling rho_D_inf", p1["ceiling_rho_D"], round(rho_inf, 4), 1e-9)
check(1, "second-order signal c", p1["c_gamma_Psi"], round(c, 4), 1e-9)
check(1, "c required for rho_D = 0.90", p1["c_required_for_target"],
      round(required_second_order_signal(0.90), 4), 1e-9)
check(1, "rho_D at 400 indicators, lambda 0.85", 0.877594, second_order_determinacy(400, 0.85), 1e-5)

for lam in (0.55, 0.65, 0.75, 0.85):
    row = p1.get(f"lambda={lam}") if FIXTURE_DIR is not None else None
    if row is None:
        continue
    for m in (4, 8, 40):
        check(1, f"rho_D at m={m}, lambda={lam}", row[f"rho_D_at_m{m}"],
              round(second_order_determinacy(m, lam), 4), 1e-9)

print(f"{len(CHECKS)} checks so far")

In [ ]:
if FIXTURE_DIR is not None:
    ident = published("p1_identification")
    for record in ident["planned_designs"]:
        rep = identification_report(record["loading"], record["label"])
        check(1, f"Jacobian rank, loading {record['loading']}", record["rank"], rep["rank"], 0)
        check(1, f"smallest singular value, loading {record['loading']}",
              record["smallest_singular_value"], rep["smallest_singular_value"], 1e-8)
    control_b = ident["controls"][1]
    rep = identification_report(0.75, control_b["label"], np.array([0.80, 0.70]),
                                np.array([0.36, 0.51]), per_domain=4)
    check(1, "negative control rank (must be short)", control_b["rank"], rep["rank"], 0)
    print("identification checks added")
else:
    print("identification fixtures unavailable under the embedded fallback; skipped")

In [ ]:
start = time.time()
res1 = paper1_validation.run()          # published configuration, 150 replicates
print(f"paper 1 full run: {time.time() - start:.1f} s")

if FIXTURE_DIR is not None:
    samples = published("p1_samplesize")["summary"]
    for key, expected in samples.items():
        actual = res1["sample_size"]["summary"].get(key)
        if actual is None:
            continue
        check(1, f"determinacy, {key}", expected["determinacy"], actual["determinacy"], 0.01,
              "simulation")
        check(1, f"worst coverage, {key}", expected["min_coverage"], actual["min_coverage"], 0.06,
              "simulation")
print(f"{len(CHECKS)} checks so far")

## Paper 2: attenuation and events

In [ ]:
from hpotential.analyses import paper2_prognostic
from hpotential.core.prognostic import events_inflation

p2 = published("p2_pipeline")
start = time.time()
res2 = paper2_prognostic.run()
print(f"paper 2 full run: {time.time() - start:.1f} s")

check(2, "rho_D of the planned design", p2["rho_D_population"],
      round(res2["rho_D_population"], 4), 1e-9)
check(2, "predicted attenuated beta", p2["predicted_attenuated_beta"],
      round(res2["predicted_attenuated_beta"], 4), 1e-9)
check(2, "events inflation at the ceiling", 1.2977, events_inflation(0.877819), 1e-4)
check(2, "events inflation at the planned design", 1.4305, events_inflation(0.8361), 1e-4)
check(2, "pooled attenuation ratio", 0.843, res2["attenuation_ratio_pooled"], 0.02, "simulation")

if FIXTURE_DIR is not None:
    for key, expected in p2.items():
        if not isinstance(expected, dict) or "attenuation_ratio" not in expected:
            continue
        actual = res2.get(key)
        if actual is None:
            continue
        check(2, f"attenuation ratio, {key}", expected["attenuation_ratio"],
              actual["attenuation_ratio"], 0.06, "simulation")
print(f"{len(CHECKS)} checks so far")

## Paper 3: Delphi operating characteristics

In [ ]:
from hpotential.analyses import paper3_consensus
from hpotential.core.consensus import iqr_lemma_violations

p3 = published("p3_delphi_oc")
start = time.time()
res3 = paper3_consensus.run()
print(f"paper 3 full run: {time.time() - start:.1f} s")

for panel, expected in p3["operating_characteristics"].items():
    actual = res3["operating_characteristics"][panel]
    check(3, f"false-drop rate, panel {panel}", expected["false_drop_at_pi_0_90"],
          actual["false_drop_at_pi_0_90"], 0.02, "simulation")
    if "false_keep_at_pi_0_65" in expected:
        check(3, f"false-keep rate, panel {panel}", expected["false_keep_at_pi_0_65"],
              actual["false_keep_at_pi_0_65"], 0.02, "simulation")

lemma = iqr_lemma_violations(max_panel=10, cvi_min=0.78, iqr_bound=2.0)
check(3, "dispersion lemma violations (linear)", 0, lemma["violations"], 0)
check(3, "largest IQR among passing panels", 2.0, lemma["max_iqr_among_passing"], 1e-12)
print(f"{len(CHECKS)} checks so far")

## Paper 4: solver verification

In [ ]:
from hpotential.analyses import paper4_dynamics

p4 = published("p4_numerics")
start = time.time()
res4 = paper4_dynamics.run()
print(f"paper 4 full run: {time.time() - start:.1f} s")

check(4, "RK4 observed order", p4["rk4"]["observed_order"], res4["rk4"]["observed_order"], 0.02)
check(4, "Euler-Maruyama strong order, additive",
      p4["euler_maruyama"]["additive_noise"]["strong_order"],
      res4["euler_maruyama"]["additive_noise"]["strong_order"], 0.05, "simulation")
check(4, "Euler-Maruyama strong order, multiplicative",
      p4["euler_maruyama"]["multiplicative_noise"]["strong_order"],
      res4["euler_maruyama"]["multiplicative_noise"]["strong_order"], 0.08, "simulation")
check(4, "Fokker-Planck L1 grid convergence order",
      p4["fokker_planck"]["grid_convergence_order_L1"],
      res4["fokker_planck"]["grid_convergence_order_L1"], 0.02)
worst_drift = max(abs(v["mass_drift"]) for k, v in res4["fokker_planck"].items()
                  if isinstance(v, dict))
check(4, "Fokker-Planck mass drift (must be at machine precision)", 0.0, worst_drift, 1e-14)
print(f"{len(CHECKS)} checks so far")

## Paper 5: the costed rule

In [ ]:
from hpotential.analyses import paper5_access

p5 = published("p5_costed")
start = time.time()
res5 = paper5_access.run()
print(f"paper 5 full run: {time.time() - start:.1f} s")

check(5, "closed form vs direct evaluation", 0.0,
      res5["closed_form_check"]["max_relative_discrepancy"], 1e-12)
for setting, expected in p5["psa"].items():
    actual = res5["psa"][setting]
    check(5, f"rules disagree, {setting}", expected["P_disagreement"],
          actual["P_disagreement"], 0.01, "simulation")
    check(5, f"median V, {setting}", expected["median_V"], actual["median_V"], 0.30, "simulation")
print(f"{len(CHECKS)} checks so far")

## Paper 6: early assessment and value of information

In [ ]:
from hpotential.analyses import paper6_hta

p6 = published("p6_voi")
start = time.time()
res6 = paper6_hta.run()
print(f"paper 6 full run: {time.time() - start:.1f} s")

for scenario, expected in p6["scenarios"].items():
    actual = res6["scenarios"][scenario]
    check(6, f"P(cost-effective, QALY), {scenario}", expected["P_cost_effective_QALY"],
          actual["P_cost_effective_QALY"], 0.01, "simulation")
    if "per_person_EVPI" in expected:
        check(6, f"per-person EVPI, {scenario}", expected["per_person_EVPI"],
              actual["per_person_EVPI"], max(300.0, 0.03 * expected["per_person_EVPI"]),
              "simulation")
    if "mean_incremental_bmp" in expected:
        check(6, f"BMP over QALY ratio, {scenario}",
              expected["mean_incremental_bmp"] / expected["mean_incremental_qaly"],
              actual["mean_incremental_bmp"] / actual["mean_incremental_qaly"], 0.05, "simulation")
print(f"{len(CHECKS)} checks so far")

## The table

In [ ]:
frame = report()

In [ ]:
failures = frame[frame["result"] == "FAIL"]
if len(failures) == 0:
    print("Every published value reproduced within tolerance.")
else:
    print(f"{len(failures)} value(s) did not reproduce. A failure means one of two things:")
    print("the package changed what it computes, or a published number was wrong.")
    print("Both need a human. Widening a tolerance is not a fix.")
    print()
    print(failures.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.0))
for i, kind in enumerate(sorted(frame["kind"].unique())):
    part = frame[frame["kind"] == kind]
    ratio = np.where(part["tol"] > 0, part["abs diff"] / part["tol"].replace(0, np.nan), np.nan)
    ax.scatter(range(len(part)), np.clip(ratio, 1e-12, None), s=22, color=series_color(i),
               label=kind, alpha=0.8)
ax.axhline(1.0, ls="--", lw=1.1, color=INK)
ax.set_yscale("log")
ax.set_xlabel("check")
ax.set_ylabel("absolute difference as a share of its tolerance")
ax.set_title("Everything under the dashed line passed", fontsize=10)
ax.legend(fontsize=8)
plt.show()

---

## What this does not establish

**A green table means the code is stable, not that the science is right.** Every check here
compares this package against numbers this package produced earlier. It is a regression
test, not an external validation.

**Simulation tolerances are judgement calls.** They are set wide enough for Monte Carlo
error at the published replicate counts and narrow enough that a change of method fails
them. A pass inside a wide tolerance is weaker evidence than a pass inside a tight one, and
the `kind` column tells you which you are looking at.

**The published values themselves rest on synthetic data.** Reproducing them exactly says
nothing about patients, panels or health systems.

**Reproducing a number is not agreeing with it.** If a value here is wrong in the papers,
this notebook will faithfully reproduce the wrong value and print PASS.